# Portfolio Optimization via FunSearch and LSTM-PPO
**Project Summary**: This project implements an evolutionary search for investment strategies. It leverages a pre-trained LSTM-PPO model for feature extraction and dynamic rebalancing, while using FunSearch (LLM-driven evolution) to discover high-performing strategy formulations.

**Reproduction Goal**: This notebook reproduces the full backtest comparison between Equal Weight, Min-Variance, Max-Sharpe, LSTM-PPO, and the Evolved FunSearch strategy using real historical data and pre-trained weights.

### 1. Environment Setup
We clone the project and install dependencies. 
**Note**: Please ensure your `data/processed` and `model_ckpt` folders are uploaded to your GitHub repository.

In [ ]:
import os
import sys

# 1. Clone your real repository
# !git clone https://github.com/your_username/AIIS.git /content/AIIS
# %cd /content/AIIS

# 2. Install all required dependencies
!pip install akshare gymnasium stable-baselines3 openai PyYAML absl-py scipy transformers accelerate

# 3. Configure Pathing
sys.path.append('/content/AIIS')
sys.path.append('/content/AIIS/funsearch')

# Verify core files exist
paths = [
    'run_funsearch_enhanced.py',
    'funsearch_specification_enhanced.py',
    'model_ckpt/best_lstm_multi_asset.pth'
]
for p in paths:
    if os.path.exists(p):
        print(f"✅ Found: {p}")
    else:
        print(f"❌ Missing: {p}")

### 2. Loading the Strategy Specification
We load the actual logic from `funsearch_specification_enhanced.py`. This file contains the backtesting engine, the data loading logic for the 5-stock portfolio, and the PPO model integration.

In [ ]:
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# Import the actual backtest and model logic from your source
import funsearch_specification_enhanced as fs

print(f"✅ Data loaded. Price array shape: {fs.price_array.shape}")
print(f"✅ LSTM Model loaded onto: {next(fs.lstm_model.parameters()).device}")

### 3. Executing the Full Evaluation
Instead of random data, we now execute the actual `backtest` functions for each strategy defined in your project. This includes the dynamic LSTM-PPO backtest and the benchmark strategies.

In [ ]:
# 1. Define storage for results
nav_dict = {}
metrics_dict = {}

# --- Strategy 1: Equal Weight ---
weights_equal = np.ones(len(fs.stock_list)) / len(fs.stock_list)
nav_eq, _, sr_eq, so_eq, mdd_eq = fs.backtest(weights_equal, fs.price_array)
nav_dict['Equal Weight'] = nav_eq
metrics_dict['Equal Weight'] = {'Sharpe': sr_eq, 'MaxDD': mdd_eq}

# --- Strategy 2: Min-Variance ---
weights_mv = fs.minvar_weights(fs.price_array)
nav_mv, _, sr_mv, so_mv, mdd_mv = fs.backtest(weights_mv, fs.price_array)
nav_dict['Min-Variance'] = nav_mv
metrics_dict['Min-Variance'] = {'Sharpe': sr_mv, 'MaxDD': mdd_mv}

# --- Strategy 3: LSTM+PPO Dynamic RL ---
print("Running Dynamic LSTM-PPO Backtest... this may take a moment.")
nav_rl, sr_rl, so_rl, _ = fs.lstm_ppo_dynamic_backtest(
    fs.price_array, 
    fs.multi_factor_array, 
    fs.ppo_model, 
    window_size=20
)
# Calculate MDD for RL
peak = np.maximum.accumulate(nav_rl)
mdd_rl = np.min((nav_rl - peak) / (peak + 1e-8))
nav_dict['LSTM+PPO Dynamic'] = nav_rl
metrics_dict['LSTM+PPO Dynamic'] = {'Sharpe': sr_rl, 'MaxDD': mdd_rl}

print("✅ All strategies evaluated using real project logic.")

### 4. Performance Visualization
We generate the cumulative NAV curves to compare how each strategy performed over the historical period.

In [ ]:
plt.figure(figsize=(14, 7))
for name, nav in nav_dict.items():
    plt.plot(nav, label=f"{name} (Sharpe: {metrics_dict[name]['Sharpe']:.2f})")

plt.title("Reproduction of Strategy Performance", fontsize=16)
plt.xlabel("Trading Days")
plt.ylabel("Net Asset Value (NAV)")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

# Print metrics table
print("\n" + "="*30)
print(f"{'Strategy':<20} | {'Sharpe':<8} | {'MaxDD':<8}")
print("-" * 40)
for name, m in metrics_dict.items():
    print(f"{name:<20} | {m['Sharpe']:<8.4f} | {m['MaxDD']:<8.4f}")